In [ ]:
from typing import Optional, List
# FRIDAY — ONE-CELL GOOGLE COLAB BACKEND
# Required Colab Secrets: NGROK_AUTHTOKEN, FRIDAY_API_KEY
# Optional Colab Secret: HF_TOKEN (for gated models such as Meta Llama)

import sys
import subprocess

# 1. Install what the runtime needs (reusable after Colab resets).
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "transformers>=4.46,<5", "accelerate>=1.0",
        "fastapi", "uvicorn", "pyngrok", "pymupdf", "pillow",
    ],
    check=True,
)

import gc
import socket
import threading
import time
from contextlib import suppress

import torch
from fastapi import FastAPI, Header, HTTPException
from fastapi.concurrency import run_in_threadpool
from pydantic import BaseModel, Field
from pyngrok import ngrok
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
import base64
import io
import re
import fitz
from PIL import Image
from google.colab import userdata

# 2. MODEL CONFIG — only this ID needs changing for a compatible instruct model.
MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
MAX_NEW_TOKENS = 256

# 3. PERSONA — edit here whenever you want to evolve Friday's personality.
FRIDAY_SYSTEM_PROMPT = """You are Friday, a personal AI assistant.
You are helpful, practical, friendly, reliable, and calm.
Automatically respond in the language the user uses. If they use Hinglish,
respond in natural Hinglish. If they use English or Hindi, follow that language.
Give clear, useful answers. Be concise by default, but explain technical topics
step by step when helpful. For code, prioritize correctness and runnable examples.
If you don't know something, admit uncertainty instead of inventing facts.
Do not pretend to have tools, access, memory, or capabilities you don't have.
""".strip()

# 4. SECRETS (enable notebook access for each secret in Colab's key icon).
def get_secret(name, *, required=False):
    try:
        value = userdata.get(name)
        if value and value.strip():
            return value.strip()
    except Exception:
        pass
    if required:
        raise RuntimeError(
            f"Missing Colab Secret: {name}. Add it in the Secrets sidebar "
            "and enable notebook access."
        )
    return None

NGROK_AUTHTOKEN = get_secret("NGROK_AUTHTOKEN", required=True)
API_KEY = get_secret("FRIDAY_API_KEY", required=True)
HF_TOKEN = get_secret("HF_TOKEN")  # optional

if not torch.cuda.is_available():
    raise RuntimeError("GPU not available. Select Runtime > Change runtime type > T4 GPU.")

# If this cell is re-run in the same runtime, close our previous tunnel/server.
if "_friday_server" in globals():
    _friday_server.should_exit = True
    _previous_thread = globals().get("_friday_thread")
    if _previous_thread and _previous_thread.is_alive():
        _previous_thread.join(timeout=10)
    _previous_tunnel = globals().get("_friday_tunnel")
    if _previous_tunnel:
        with suppress(Exception):
            ngrok.disconnect(_previous_tunnel.public_url)
    for _name in ("_friday_model", "_friday_processor"):
        if _name in globals():
            del globals()[_name]
    gc.collect()
    torch.cuda.empty_cache()

# 5. LOAD MODEL
print("Loading:", MODEL_ID)
_friday_processor = AutoProcessor.from_pretrained(
    MODEL_ID, token=HF_TOKEN, min_pixels=256*28*28, max_pixels=768*28*28
)
_friday_model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    torch_dtype=torch.float16,  # T4-compatible; may need changes for other GPUs/models
    device_map="auto",
)
_friday_model.eval()

# Single generation at a time avoids overlapping GPU requests.
_generation_lock = threading.Lock()

def prepare_files(attachments, question):
    images, sections, notices = [], [], []
    terms = set(re.findall(r"[\w]+", question.lower())) - {"what", "the", "this", "that", "from", "with", "have", "about", "please", "document", "pdf"}
    for attachment in attachments:
        try:
            raw = base64.b64decode(attachment.data_url.split(",", 1)[1], validate=True)
            if len(raw) > 20 * 1024 * 1024:
                raise ValueError("File exceeds 20 MB")
            if attachment.type.startswith("image/"):
                img = Image.open(io.BytesIO(raw))
                img.thumbnail((1600, 1600))
                images.append(img.convert("RGB"))
                sections.append(f"Image supplied: {attachment.name}")
            elif attachment.type == "application/pdf":
                with fitz.open(stream=raw, filetype="pdf") as doc:
                    if doc.needs_pass:
                        raise ValueError("Password-protected PDFs are not supported")
                    if len(doc) > 200:
                        raise ValueError("PDF exceeds the 200-page limit")
                    pages = [(i + 1, page.get_text("text").strip()) for i, page in enumerate(doc)]
                    searchable = [(n, text) for n, text in pages if len(text) >= 40]
                    if searchable:
                        ranked = sorted(searchable, key=lambda item: sum(item[1].lower().count(t) for t in terms if len(t) > 2), reverse=True)
                        chosen = sorted(ranked[:8], key=lambda item: item[0])
                        remaining = 18000
                        for number, text in chosen:
                            if remaining <= 0: break
                            excerpt = text[:min(3000, remaining)]
                            sections.append(f"PDF {attachment.name}, page {number}:\n{excerpt}")
                            remaining -= len(excerpt)
                        if len(searchable) > len(chosen) or any(len(t) > 3000 for _, t in chosen) or sum(min(len(t), 3000) for _, t in chosen) > 18000:
                            notices.append(f"{attachment.name}: selected relevant excerpts, not the entire PDF; ask about specific pages or topics for more detail.")
                        if len(searchable) < len(pages):
                            notices.append(f"{attachment.name}: some pages have little selectable text and were not analyzed visually.")
                    else:
                        for page in list(doc)[:3]:
                            pix = page.get_pixmap(matrix=fitz.Matrix(1.2, 1.2), alpha=False)
                            images.append(Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB"))
                        notices.append(f"{attachment.name}: scanned PDF; only the first {min(len(doc), 3)} page(s) were sent as images.")
                    if not len(doc): raise ValueError("PDF has no pages")
            else:
                raise ValueError("Unsupported file type")
        except Exception as exc:
            raise ValueError(f"Could not read {attachment.name}: {exc}") from exc
    return images, sections, notices

def generate_response(message, history, attachments):
    images, sections, notices = prepare_files(attachments, message)
    if len(images) > 6:
        raise ValueError("Too many image pages to analyze at once. Attach fewer files.")
    messages = [{"role": "system", "content": FRIDAY_SYSTEM_PROMPT}]
    messages.extend({"role": turn.role, "content": turn.content} for turn in history[-12:] if turn.content.strip())
    content = [{"type": "image"} for _ in images]
    content.append({"type": "text", "text": ("\n\n".join(sections) + "\n\n" if sections else "") + "User question: " + message})
    messages.append({"role": "user", "content": content})
    prompt = _friday_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = _friday_processor(text=[prompt], images=images or None, padding=True, return_tensors="pt").to(_friday_model.device)
    with _generation_lock, torch.inference_mode():
        output_ids = _friday_model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
            pad_token_id=_friday_processor.tokenizer.pad_token_id or _friday_processor.tokenizer.eos_token_id)
    reply_tokens = output_ids[0, inputs["input_ids"].shape[-1]:]
    return _friday_processor.tokenizer.decode(reply_tokens, skip_special_tokens=True).strip(), notices

app = FastAPI(title="Friday Colab Backend")

class Turn(BaseModel):
    role: str
    content: str = Field(max_length=3000)

class FileAttachment(BaseModel):
    name: str = Field(max_length=255)
    type: str
    data_url: str

class ChatRequest(BaseModel):
    message: str = Field(min_length=1, max_length=4000)
    history: List[Turn] = Field(default_factory=list, max_length=12)
    attachments: List[FileAttachment] = Field(default_factory=list, max_length=4)

@app.get("/health")
def health():
    return {"status": "running", "model": MODEL_ID, "protocol": 2}

@app.post("/chat")
async def chat(data: ChatRequest, x_api_key: str = Header()):
    import secrets
    if not secrets.compare_digest(x_api_key, API_KEY):
        raise HTTPException(status_code=401, detail="Unauthorized")
    if not data.message.strip() or any(t.role not in ("user", "assistant") for t in data.history):
        raise HTTPException(status_code=422, detail="Invalid message or conversation history")
    try:
        result, notices = await run_in_threadpool(generate_response, data.message.strip(), data.history, data.attachments)
    except ValueError as exc:
        raise HTTPException(status_code=422, detail=str(exc)) from exc
    return {"response": result, "notices": notices, "protocol": 2}

# 7. UVICORN — select a free port to avoid colliding with earlier Colab cells.
with socket.socket() as _sock:
    _sock.bind(("127.0.0.1", 0))
    PORT = _sock.getsockname()[1]

_friday_server = __import__("uvicorn").Server(
    __import__("uvicorn").Config(
        app, host="127.0.0.1", port=PORT, log_level="warning"
    )
)
_friday_thread = threading.Thread(
    target=_friday_server.run, daemon=True
)
_friday_thread.start()

for _ in range(100):
    if _friday_server.started:
        break
    if not _friday_thread.is_alive():
        raise RuntimeError("FastAPI failed to start. Check runtime logs.")
    time.sleep(0.1)
else:
    raise RuntimeError("FastAPI startup timed out.")

# 8. EXPOSE THROUGH NGROK
ngrok.set_auth_token(NGROK_AUTHTOKEN)
_friday_tunnel = ngrok.connect(addr=PORT, proto="http")

print("\nFRIDAY IS READY")
print("Model:", MODEL_ID)
print("Ngrok URL:", _friday_tunnel.public_url)
print("Health:", _friday_tunnel.public_url + "/health")
print("API key: loaded securely from Colab Secrets (not printed).")
print("Keep Colab connected; update NGROK_BASE_URL if this URL changes.")
